# 02 — Source download and resume

Owner: Member 2. Runs on Google Colab CPU. Takes the paper IDs and version counts found by
`01_discovery.ipynb`, downloads every source version from `export.arxiv.org/src/{id}vN`, keeps
**only `.tex` and `.bib` files**, and saves progress on Google Drive so a disconnect loses nothing.

Functions provided to the rest of the pipeline:

| Function | Returns |
|---|---|
| `assigned_ids()` | `list[str]`: the 20,000 IDs in `ID_RANGES` |
| `load_manifest(path)` / `validate_assigned_manifest(records)` | normalized discovery records (`arxiv_id`, `version_count`, ...) |
| `extract_source(archive, folder)` | writes the `.tex`/`.bib` files of one archive; returns sizes and checksums |
| `download_sources(manifest, output, state, ...)` | downloads / resumes every version; returns the run summary |

Output: `DATA_DIR/<yymm-id>/vN/` (only `.tex`/`.bib`). State: `STATE_DIR/download/`.

This notebook only defines the downloader and its offline tests, like `01_discovery.ipynb`.
The full crawl is started from `05_run_pipeline.ipynb`.

### How it works

1. **One paced connection.** `SourceClient` sends one request at a time, at least `ARXIV_DELAY`
   (3 s) apart, retries 5xx with backoff and honours `Retry-After`. On HTTP 429 it moves on to the next
   version and asks for the refused one again later in the run, with the requests spaced further apart.
2. **Pipeline.** The main thread only downloads, to the fast local disk. Extraction and the slow
   writes to Drive run in `DOWNLOAD_WORKERS` background threads, so they overlap the 3 s wait
   instead of adding to it. At most `2 × DOWNLOAD_WORKERS` fetched archives wait for processing.
3. **Only `.tex` and `.bib`.** Archives are unpacked on local disk and only `.tex`/`.bib` are copied
   to Drive. Figures, `.bbl`, `.sty`, `.cls` and the raw archive never reach Drive. A version with no
   `.tex` (PDF-only) gets an empty `vN/` folder and an `empty_source` receipt.
4. **Fast resume.** Every finished version gets a receipt (`receipts/<id>/vN.json`, also appended
   to `receipts.jsonl`). At start-up the index is read once and trusted (`DOWNLOAD_VERIFY = "receipt"`):
   checking the files themselves costs ~17 Drive calls per version, hours for the whole corpus.
   `"size"` checks every saved file by size and `"sha256"` re-hashes it, e.g. once after a session
   that Colab killed before Drive finished syncing. Files other than `.tex`/`.bib` kept by older runs
   are deleted during the check, without new requests.

**Error handling and order.** A failed version is written to `errors.jsonl` and retried on the next run.
Every request is logged in `attempts.jsonl` before it is sent, so each run starts with the versions never
requested; versions requested before but not finished (404, throttled, cut, or the run was stopped or
killed mid-download) come after them, the least-tried first.
Unsafe archive paths (`../`, links, a file used as a folder) and HTML/JSON error pages are rejected.
Files whose names differ only in case are both kept (renamed `name~2.tex` on a case-insensitive disk).
On Ctrl+C or a Colab stop, archives that were already downloaded are processed before exiting.

In [ ]:
if "CONFIG_LOADED" not in globals():
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")

import bz2
import errno
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from email.utils import parsedate_to_datetime
import gzip
import hashlib
import lzma
from pathlib import PurePosixPath
import shutil
import socket
import tarfile
import tempfile
import threading
import uuid
import zipfile
from datetime import timedelta
from zoneinfo import ZoneInfo

import requests

try:
    from tqdm.auto import tqdm   # progress bars (widgets in Jupyter, text on Colab and elsewhere)
except ImportError:
    tqdm = None                  # no bars: plain progress lines instead
HTML_BARS = False   # True on Colab: download_sources draws its bars as HTML (_HtmlBars), not with tqdm
try:
    import google.colab  # noqa: F401
    from IPython import get_ipython
    # Colab's widget bars (tqdm.notebook) may not render at all (ipywidgets 8) and are lost for good when
    # the browser reconnects to a running cell (e.g. after a night). The HTML bars are drawn again in a new
    # output every few minutes, so they come back; tqdm's text bars stay the fallback outside a notebook.
    HTML_BARS = get_ipython() is not None
    if tqdm is not None:
        from tqdm.std import tqdm
except ImportError:
    pass                         # not on Colab
# Text bars get one line only: a second text bar is printed again on every redraw in notebooks.
TEXT_BARS = tqdm is not None and not any(c.__module__ == "tqdm.notebook" for c in tqdm.__mro__)

try:
    import psutil
except ImportError:  # The downloader still works without RAM sampling.
    psutil = None

## 1. Settings

`ARXIV_CONTACT` (optional, an email address) is added to the User-Agent. `DOWNLOAD_WORKERS` and
`DOWNLOAD_VERIFY` are explained in the introduction.

In [ ]:
KEEP = frozenset({".tex", ".bib"})      # the only files written to Drive
ID_PATTERN = re.compile(r"^\d{2}(?:0[1-9]|1[0-2])\.\d{4,5}$")
RETRY_STATUSES = frozenset({429, 500, 502, 503, 504})
THROTTLE_STATUSES = frozenset({429})    # "too many requests": move on, ask for that version again later
THROTTLE_REQUEUES = 3                   # times a 429 version goes back to the end of the same run
MAX_TEXT_BYTES = 128 * 1024 * 1024       # safety limit for one .tex/.bib file
TEX_COMMAND = re.compile(rb"\\(?:documentclass|documentstyle|begin|input|def|newcommand|section)\b")

DOWNLOAD_STATE = STATE_DIR / "download"   # receipts, errors, progress, papers.csv
DOWNLOAD_WORKERS = 2        # threads that extract archives and write to Drive
LOG_TIMEZONE = "Asia/Ho_Chi_Minh"   # time shown in front of FAILED / Papers lines (Colab runs in UTC)
DOWNLOAD_VERIFY = "receipt"   # resume check: "receipt" (trust receipts.jsonl, no Drive scan),
                              # "size" or "sha256" (check every saved file; slow on Drive)

_contact = os.environ.get("ARXIV_CONTACT", "").strip()
if _contact and "@" not in _contact:
    raise ValueError("ARXIV_CONTACT must be an email address when set.")
DOWNLOAD_USER_AGENT = USER_AGENT + (f" (mailto:{_contact})" if _contact else "")
DOWNLOAD_STATE.mkdir(parents=True, exist_ok=True)

## 2. Discovery manifest

A JSON list (or `{"papers": [...]}`) with `id`/`versions` from `01_discovery`; `arxiv_id`/`version_count`
is also accepted. Duplicate IDs and impossible version counts are rejected.

In [ ]:
def _now():
    return datetime.now(timezone.utc).isoformat()


def _atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        with temporary.open("w", encoding="utf-8") as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def validate_manifest(records):
    """Normalize discovery records, rejecting ambiguous IDs and counts."""
    if not isinstance(records, list) or not records:
        raise ValueError("Manifest must contain a nonempty list of papers")
    normalized, seen = [], set()
    for item in records:
        if not isinstance(item, dict):
            raise ValueError("Each paper must be an object")
        identifier = item.get("arxiv_id", item.get("id"))
        count = item.get("version_count", item.get("versions"))
        if not isinstance(identifier, str) or not ID_PATTERN.fullmatch(identifier) or int(identifier.split(".")[1]) == 0:
            raise ValueError(f"Invalid modern arXiv ID: {identifier!r}")
        if identifier in seen:
            raise ValueError(f"Duplicate arXiv ID: {identifier}")
        if type(count) is not int or count < 0 or (count == 0 and item.get("status") not in ("not_found", "error")):
            raise ValueError(f"Invalid version count for {identifier}: {count!r}")
        seen.add(identifier)
        normalized.append(dict(item, arxiv_id=identifier, version_count=count))
    return normalized


def load_manifest(path):
    with Path(path).open(encoding="utf-8-sig") as stream:
        document = json.load(stream)
    return validate_manifest(document.get("papers") if isinstance(document, dict) else document)


def assigned_ids():
    """The exact 20,000 IDs in ID_RANGES (00_config), in instructor order."""
    ids = []
    for start, end in ID_RANGES:
        month, low = start.split(".")
        ids += [f"{month}.{n:0{len(low)}d}" for n in range(int(low), int(end.split(".")[1]) + 1)]
    return ids


def validate_assigned_manifest(records):
    normalized = validate_manifest(records)
    if {record["arxiv_id"] for record in normalized} != set(assigned_ids()):
        raise ValueError("Discovery manifest must cover exactly the 20,000 assigned IDs")
    return normalized

## 3. Paced source client

All attempts, retries included, share one rate gate. A 5xx response pushes the next request
back by `max(2^attempt, Retry-After)` seconds; 404 and other errors are not retried.

**HTTP 429** (arXiv's rate limit: `Rate exceeded.` per IP over a short window, and Colab IPs are shared) is
not a problem of the version, so it is not marked failed. `fetch` raises `Throttled` after that one request,
and `download_sources` moves on to the next version and puts the refused one back at the end of this run's
work (up to `THROTTLE_REQUEUES` = 3 times; after that it waits for the next run). The next request often
succeeds, so the crawl keeps downloading instead of waiting. Each 429 doubles the spacing between requests
(`pace`, from `ARXIV_DELAY` up to 8 s) and each success halves it back. After 5 answers 429 in a row the
client also waits a cooldown (10 s, doubled up to 60 s, or `Retry-After` if longer), and after
`max_throttled` (50) in a row the run stops: that runtime's IP is rate-limited, switch to another one.

arXiv often cuts long transfers (`IncompleteRead`). The next attempt then asks only for the missing
bytes (`Range`, with `If-Range` on the `ETag`, so a changed file starts over) instead of the whole
archive. When a version still fails on a 5xx or a cut connection after its last attempt,
the next request waits a **cooldown**: 10 s, doubled after each such failure up to 60 s, and reset
by the next successful download, so a period of throttling costs a few pauses instead of dozens of
failed versions. A normal download never waits for it, and the resume makes such failures rarer.

In [ ]:
class Throttled(requests.HTTPError):
    """HTTP 429, arXiv's rate limit: not a failure of the version, which is asked for again later."""


class SourceClient:
    """Single-threaded arXiv source client; retry attempts share the rate gate."""

    def __init__(self, session=None, user_agent=USER_AGENT, delay=ARXIV_DELAY,
                 max_attempts=4, clock=time.monotonic, sleeper=time.sleep,
                 min_cooldown=10.0, max_cooldown=60.0, max_pace=8.0, max_throttled=50, cooldown_after=5):
        if delay < 3:
            raise ValueError("arXiv delay must be at least 3 seconds")
        if max_attempts < 1:
            raise ValueError("max_attempts must be positive")
        self.session = session if session is not None else requests.Session()
        self.session.headers.update({"User-Agent": user_agent, "Accept-Encoding": "identity"})
        self.delay, self.max_attempts = delay, max_attempts
        self.clock, self.sleeper = clock, sleeper
        self.last_start = None
        self.not_before = 0.0
        self.min_cooldown, self.max_cooldown = min_cooldown, max_cooldown
        self.cooldown = 0.0   # current pause after throttled versions; 0 after a success
        self.pause = 0.0      # cooldown started by the last fetch (0 if none), for the log
        self.on_wait = None   # on_wait(seconds): called before a cooldown or Retry-After wait (not the pacing)
        self.pace = delay     # seconds between request starts: doubled on each 429, halved after a success
        self.max_pace = max_pace
        self.throttled, self.max_throttled = 0, max_throttled   # answers 429 in a row (any version)
        self.cooldown_after = cooldown_after   # 429s in a row before the client also waits a cooldown
        self.on_throttle = None   # on_throttle(label, seconds): a 429; seconds = wait before the next request

    def _pace(self):
        wait = max(0.0, self.not_before - self.clock())
        if self.last_start is not None:
            wait = max(wait, self.pace - (self.clock() - self.last_start))
        if wait:
            if self.on_wait is not None and wait > max(self.delay, self.max_pace):
                self.on_wait(wait)
            self.sleeper(wait)
        self.last_start = self.clock()

    def _cool_down(self):
        """The last attempt was throttled or cut: wait longer before the next request (doubles up to the max)."""
        self.cooldown = min(self.max_cooldown, max(self.min_cooldown, 2 * self.cooldown))
        self.not_before = max(self.not_before, self.clock() + self.cooldown)
        self.pause = self.cooldown

    def _throttle(self, label, retry_after):
        """HTTP 429: space the next requests further apart; after several in a row, also wait a cooldown."""
        self.throttled += 1
        self.pace = min(max(self.max_pace, self.delay), 2 * self.pace)   # never closer than delay
        wait = self._retry_after(retry_after)
        if self.throttled >= self.cooldown_after:   # arXiv refuses everything for now
            self.cooldown = min(self.max_cooldown, max(self.min_cooldown, 2 * self.cooldown))
            wait = max(wait, self.cooldown)
        self.not_before = max(self.not_before, self.clock() + wait)
        if self.on_throttle is not None:
            self.on_throttle(label, wait)

    @staticmethod
    def _range_start(value):
        match = re.match(r"bytes (\d+)-", value or "")
        return int(match.group(1)) if match else None

    @staticmethod
    def _retry_after(value):
        if not value:
            return 0.0
        try:
            return max(0.0, float(value))
        except ValueError:
            try:
                return max(0.0, (parsedate_to_datetime(value) - datetime.now(timezone.utc)).total_seconds())
            except (TypeError, ValueError, OverflowError):
                return 0.0

    def fetch(self, arxiv_id, version, destination, progress=None):
        """Stream one version into a temporary file, then replace destination.

        progress(done_bytes, total_bytes) is called while streaming (total is None when the
        server sends no Content-Length); it drives the "MB left" bar of download_sources.
        A transfer cut midway is resumed on the next attempt with an HTTP Range request.
        """
        if not ID_PATTERN.fullmatch(arxiv_id) or type(version) is not int or version < 1:
            raise ValueError("Invalid arXiv ID/version")
        destination = Path(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://export.arxiv.org/src/{arxiv_id}v{version}"
        self.pause = 0.0
        # The raw archive is a throwaway file on local runtime disk:
        # no fsync, it is extracted and deleted right after.
        temporary = destination.with_name(destination.name + "." + uuid.uuid4().hex + ".part")
        etag = total = None
        attempt = 0   # failed attempts (a 429 is not retried here: it raises Throttled at once)
        try:
            while True:
                self._pace()
                size = temporary.stat().st_size if etag and temporary.exists() else 0
                headers = {"Range": f"bytes={size}-", "If-Range": etag} if size else {}
                response = None
                try:
                    response = self.session.get(url, stream=True, timeout=(10, 120), allow_redirects=False,
                                                headers=headers)
                    status = response.status_code
                    if status == 206 and size and self._range_start(response.headers.get("Content-Range")) != size:
                        etag = None   # not the missing bytes: start over on the next attempt
                        raise requests.RequestException(f"Range answer does not continue the partial file from {url}")
                    resumed = status == 206 and bool(size)
                    if status != 200 and not resumed:
                        error = requests.HTTPError(f"HTTP {status} from {url}", response=response)
                        if status in THROTTLE_STATUSES:
                            self._throttle(f"{arxiv_id}v{version}", response.headers.get("Retry-After"))
                            raise Throttled(f"HTTP {status} from {url}", response=response)
                        attempt += 1
                        last = attempt >= self.max_attempts
                        if status in RETRY_STATUSES:
                            self.not_before = max(self.not_before,
                                                  self.clock() + max(2 ** (attempt - 1), self._retry_after(response.headers.get("Retry-After"))))
                            if last:
                                self._cool_down()
                        if status not in RETRY_STATUSES or last:
                            raise error
                        continue
                    if not resumed:   # a full answer: (re)start the file
                        size, etag = 0, response.headers.get("ETag")
                    length = response.headers.get("Content-Length", "")
                    total = size + int(length) if length.isdigit() else None
                    if progress:
                        progress(size, total)
                    with temporary.open("ab" if resumed else "wb") as output:
                        for block in response.iter_content(chunk_size=1024 * 1024):
                            if block:
                                output.write(block)
                                size += len(block)
                                if progress:
                                    progress(size, total)
                    if not size:
                        raise requests.RequestException(f"Empty source response from {url}")
                    os.replace(temporary, destination)
                    self.cooldown = 0.0
                    self.throttled = 0
                    self.pace = max(self.delay, 0.5 * self.pace)   # back towards the normal spacing
                    return size
                except requests.HTTPError:
                    raise
                except (requests.RequestException, OSError) as exc:
                    attempt += 1
                    if attempt >= self.max_attempts:
                        if isinstance(exc, requests.RequestException):
                            self._cool_down()
                        raise
                    self.sleeper(2 ** (attempt - 1))
                finally:
                    if response is not None:
                        response.close()
        finally:
            temporary.unlink(missing_ok=True)

## 4. Keep only `.tex` and `.bib`

`extract_source` accepts TAR (any compression), ZIP, a single gzip/bzip2/xz file, plain TeX, or a
PDF/Word file (no source). Every archive path is checked before anything is written. `before_bytes`
is the unpacked size of all files; `after_bytes` is the size of the kept `.tex`/`.bib` files.

In [ ]:
def _safe_name(name):
    raw = str(name).replace("\\", "/")
    if (not raw or raw.startswith("/") or "\x00" in raw or ":" in raw
            or any(part in ("..", "") for part in raw.split("/"))):
        raise ValueError(f"Unsafe archive path: {name!r}")
    return "/".join(part for part in raw.split("/") if part != ".")


def _check_members(entries):
    """Reject unsafe paths, and a file whose path is also used as a folder (exact names)."""
    files, folders = set(), set()
    for name, is_file in entries:
        normalized = _safe_name(name)
        if not normalized:
            if is_file:
                raise ValueError("Archive file has no path")
            continue
        if is_file:
            files.add(normalized)
            folders.update(str(p) for p in PurePosixPath(normalized).parents if str(p) != ".")
    conflicts = files & folders
    if conflicts:
        raise ValueError(f"Archive path conflicts with file: {sorted(conflicts)[0]!r}")


def _case_insensitive(directory):
    """True if `directory` treats A.tex and a.tex as one file (Windows, macOS)."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    probe = directory / f".case-probe-{uuid.uuid4().hex}"
    probe.touch()
    try:
        return probe.with_name(probe.name.upper()).exists()
    finally:
        probe.unlink(missing_ok=True)


def _store_name(name, taken, fold_case):
    """Where to store a kept file: A.tex/a.tex clashes are renamed only on case-insensitive disks."""
    if not fold_case or name.casefold() not in taken:
        return name
    path, n = PurePosixPath(name), 2
    while str(path.with_name(f"{path.stem}~{n}{path.suffix}")).casefold() in taken:
        n += 1
    return str(path.with_name(f"{path.stem}~{n}{path.suffix}"))


def _copy_text(stream, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    digest, size = hashlib.sha256(), 0
    with destination.open("xb") as output:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            size += len(block)
            if size > MAX_TEXT_BYTES:
                raise ValueError("Source text file exceeds safety limit")
            digest.update(block)
            output.write(block)
    return {"size": size, "sha256": digest.hexdigest()}


def _known_non_tex(header):
    return header.startswith((b"%PDF-", b"\xd0\xcf\x11\xe0"))


def _looks_like_markup(header):
    return header.lstrip().lower().startswith((b"<html", b"<!doctype", b"<?xml", b"{"))


def _kept(name):
    return PurePosixPath(name).suffix.lower() in KEEP


def _single_tex(stream, first, destination):
    """Stream a single-file source to main.tex in one pass, checking it is TeX."""
    target = destination / "main.tex"
    digest, size, found, tail = hashlib.sha256(), 0, False, b""
    try:
        with target.open("xb") as output:
            block = first
            while block:
                size += len(block)
                if size > MAX_TEXT_BYTES:
                    raise ValueError("Single-file source exceeds safety limit")
                found = found or bool(TEX_COMMAND.search(tail + block))
                tail = block[-64:]
                digest.update(block)
                output.write(block)
                block = stream.read(1024 * 1024)
        if not found:
            raise ValueError("Unrecognized single-file source response")
    except BaseException:
        target.unlink(missing_ok=True)
        raise
    return size, {"main.tex": {"size": size, "sha256": digest.hexdigest()}}


def _extract_members(members, open_member, destination, result, fold_case):
    """Check every path, then copy .tex/.bib only if the archive has a .tex.

    A name listed twice keeps its last copy, as tar and unzip do. Names that differ only in case
    (2609.32867v1 has sections/limitations.tex and sections/Limitations.tex, with different text)
    are different files and both are kept. On a case-insensitive disk the later one is stored as
    Limitations~2.tex and result["renamed"] maps the stored name back to the original one.
    """
    _check_members([(name.rstrip("/"), is_file) for name, is_file, _size, _m in members])
    files = {}
    for name, is_file, size, member in members:
        if is_file:
            files[_safe_name(name)] = (size, member)   # a repeated name keeps the last copy
    result["before_bytes"] = sum(size for size, _member in files.values())
    if not any(PurePosixPath(name).suffix.lower() == ".tex" for name in files):
        return
    taken = set()
    for name, (_size, member) in files.items():
        if not _kept(name):
            continue
        stored = _store_name(name, taken, fold_case)
        taken.add(stored.casefold())
        if stored != name:
            result.setdefault("renamed", {})[stored] = name
        with open_member(member) as stream:
            result["files"][stored] = _copy_text(stream, destination / stored)


def _extract_zip(archive, destination, result, fold_case):
    with zipfile.ZipFile(archive) as package:
        members = package.infolist()
        for member in members:
            mode = (member.external_attr >> 16) & 0o170000
            if mode not in (0, 0o100000, 0o040000):
                raise ValueError(f"Special ZIP member: {member.filename!r}")
        _extract_members([(m.filename, not m.is_dir(), m.file_size, m) for m in members],
                         package.open, destination, result, fold_case)


def _open_tar(archive):
    try:
        return tarfile.open(archive, mode="r:*")
    except tarfile.TarError:
        return None


def _extract_tar(package, destination, result, fold_case):
    with package:
        members = package.getmembers()
        for member in members:
            if not (member.isfile() or member.isdir()):
                raise ValueError(f"Special TAR member: {member.name!r}")
        _extract_members([(m.name, m.isfile(), m.size, m) for m in members],
                         package.extractfile, destination, result, fold_case)


def extract_source(archive, destination, fold_case=None):
    """Write only .tex/.bib; compute uncompressed member sizes without storing figures.

    fold_case: rename A.tex/a.tex clashes (default: only if `destination` is case-insensitive).
    """
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if fold_case is None:
        fold_case = _case_insensitive(destination)
    result = {"status": "empty_source", "before_bytes": 0, "after_bytes": 0, "files": {}}
    with archive.open("rb") as stream:
        header = stream.read(8192)
    if zipfile.is_zipfile(archive):
        _extract_zip(archive, destination, result, fold_case)
    elif (package := _open_tar(archive)) is not None:
        _extract_tar(package, destination, result, fold_case)
    else:
        opener = gzip.open if header.startswith(b"\x1f\x8b") else (
            bz2.open if header.startswith(b"BZh") else (
                lzma.open if header.startswith(b"\xfd7zXZ") else None))
        if opener is not None:
            with opener(archive, "rb") as stream:
                first = stream.read(8192)
                if _known_non_tex(first):
                    result["before_bytes"] = len(first) + sum(len(block) for block in iter(lambda: stream.read(1024 * 1024), b""))
                elif _looks_like_markup(first):
                    raise ValueError("Unrecognized HTML/JSON source response")
                else:
                    result["before_bytes"], result["files"] = _single_tex(stream, first, destination)
        elif _known_non_tex(header):
            result["before_bytes"] = archive.stat().st_size
        elif _looks_like_markup(header):
            raise ValueError("Unrecognized HTML/JSON source response")
        else:
            with archive.open("rb") as stream:
                result["before_bytes"], result["files"] = _single_tex(stream, stream.read(1024 * 1024), destination)
    if any(PurePosixPath(name).suffix.lower() == ".tex" for name in result["files"]):
        result["status"] = "ok"
        result["after_bytes"] = sum(entry["size"] for entry in result["files"].values())
    return result

## 5. Receipts and fast resume

A version counts as done when it has a receipt. The `receipts.jsonl` index is read in one pass; the
per-version receipt file is the fallback (e.g. after a crash between the two writes). With
`verify="receipt"` an index entry is trusted, because it is written only after the version's files were
renamed into `vN/`; with `"size"`/`"sha256"` every receipt must also match the files on disk. Each file
check costs ~10-30 ms on Drive, so `"size"` takes hours on the full corpus. Versions without any receipt
are skipped without a lookup.

In [ ]:
def _receipt_path(state_dir, arxiv_id, version):
    return Path(state_dir) / "receipts" / to_folder_id(arxiv_id) / f"v{version}.json"


def _index_path(state_dir):
    return Path(state_dir) / "receipts.jsonl"


def _past_tries(state_dir):
    """{(arxiv_id, version): times requested in earlier runs}, from attempts.jsonl (one line per request
    started, so a run stopped or killed mid-download counts too) and errors.jsonl (runs from before
    attempts.jsonl existed)."""
    tries = {}
    for name in ("attempts.jsonl", "errors.jsonl"):
        try:
            lines = (Path(state_dir) / name).read_text(encoding="utf-8", errors="replace").splitlines()
        except FileNotFoundError:
            continue
        counted = {}
        for line in lines:
            try:
                row = json.loads(line)
                key = (row["arxiv_id"], int(row["version"]))
            except (ValueError, KeyError, TypeError):
                continue   # a line torn by a killed run
            counted[key] = counted.get(key, 0) + 1
        for key, n in counted.items():
            tries[key] = max(tries.get(key, 0), n)
    return tries


def _read_receipt_index(state_dir):
    """One sequential read instead of thousands of small receipt files on Drive."""
    index = {}
    try:
        with _index_path(state_dir).open(encoding="utf-8") as stream:
            for line in stream:
                try:
                    receipt = json.loads(line)
                    index[(receipt["arxiv_id"], receipt["version"])] = receipt
                except (ValueError, KeyError, TypeError):
                    continue  # A torn final line after a crash is ignored.
    except FileNotFoundError:
        pass
    return index


def _file_digest(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _receipt_consistent(receipt, arxiv_id, version):
    """The receipt agrees with itself (no disk access): this version, a final status, a .tex if ok."""
    if (receipt["arxiv_id"] != arxiv_id or receipt["version"] != version
            or receipt["status"] not in ("ok", "empty_source")):
        return False
    files = receipt["files"]
    if receipt["status"] == "empty_source":
        return not files
    return any(PurePosixPath(n).suffix.lower() == ".tex" for n in files)


def _check_receipt(receipt, directory, arxiv_id, version, verify):
    """Compare a receipt with the files on disk (sizes; SHA-256 if verify='sha256')."""
    if not _receipt_consistent(receipt, arxiv_id, version):
        return False
    files = receipt["files"]
    actual = {}
    for root, _dirs, names in os.walk(directory):
        for name in names:
            path = Path(root) / name
            if path.is_symlink():
                return False
            actual[path.relative_to(directory).as_posix()] = path.stat().st_size
    if actual != {name: entry["size"] for name, entry in files.items()}:
        return False
    if verify == "sha256":
        return all(_file_digest(directory / name) == entry["sha256"] for name, entry in files.items())
    return True


def _prune_unkept(receipt, directory, state_dir):
    """Drop files kept by older runs (.bbl/.sty/.cls) without re-downloading."""
    extra = [name for name in receipt["files"] if not _kept(name)]
    if not extra:
        return receipt
    for name in extra:
        path = directory / name
        path.unlink(missing_ok=True)
        for parent in path.parents:
            if parent == directory or not parent.is_relative_to(directory):
                break
            try:
                parent.rmdir()
            except OSError:
                break
    files = {name: entry for name, entry in receipt["files"].items() if _kept(name)}
    receipt = dict(receipt, files=files, after_bytes=sum(entry["size"] for entry in files.values()))
    _atomic_json(_receipt_path(state_dir, receipt["arxiv_id"], receipt["version"]), receipt)
    return receipt


def verified_receipt(state_dir, output_dir, arxiv_id, version, *, verify="size", cached=None):
    """Return a receipt that matches the saved files, or None to re-download.

    ``cached`` (from receipts.jsonl) is tried first; the per-version receipt
    file is the fallback, e.g. after a crash between the two writes.
    """
    directory = Path(output_dir) / to_folder_id(arxiv_id) / f"v{version}"
    try:
        if not directory.is_dir() or directory.is_symlink():
            return None
    except OSError:
        return None
    for receipt in ([cached] if cached is not None else []) + [None]:
        if receipt is None:  # Fallback: the per-version receipt file.
            try:
                receipt = json.loads(_receipt_path(state_dir, arxiv_id, version).read_text(encoding="utf-8"))
            except (OSError, ValueError):
                return None
        try:
            if _check_receipt(receipt, directory, arxiv_id, version, verify):
                return _prune_unkept(receipt, directory, state_dir)
        except (OSError, ValueError, KeyError, TypeError, AttributeError):
            pass
    return None


def _ids_with_receipt_files(state_dir):
    """Folder names in state/receipts/ (one listing): the papers that have per-version receipt files."""
    try:
        with os.scandir(Path(state_dir) / "receipts") as entries:
            return {entry.name for entry in entries}
    except FileNotFoundError:
        return set()


def load_receipts(records, output_dir, state_dir, *, verify="size"):
    """Verify saved versions once at startup; refresh the index for pruned receipts.

    verify='receipt' trusts receipts.jsonl and reads no saved file: a receipt is written only after its
    files were renamed into vN/. Receipts found only as per-version files, or listing files that older
    runs kept, are still checked on disk. 'size' / 'sha256' check every saved version (~17 Drive calls each).
    Only versions with a receipt (receipts.jsonl or a per-version file) can pass, so the others are
    skipped without a lookup in output_dir: on Drive each lookup costs ~10-30 ms, for every assigned version.
    """
    index = _read_receipt_index(state_dir)
    indexed = {arxiv_id for arxiv_id, _version in index}
    with_files = _ids_with_receipt_files(state_dir)
    receipts, changed = {}, []
    for record in records:
        if record["arxiv_id"] not in indexed and to_folder_id(record["arxiv_id"]) not in with_files:
            continue
        for version in range(1, record["version_count"] + 1):
            key = (record["arxiv_id"], version)
            cached = index.get(key)
            if verify == "receipt" and cached is not None:
                try:
                    trusted = _receipt_consistent(cached, *key) and all(_kept(n) for n in cached["files"])
                except (KeyError, TypeError, AttributeError):
                    trusted = False
                if trusted:
                    receipts[key] = cached
                    continue
            receipt = verified_receipt(state_dir, output_dir, *key, cached=cached,
                                       verify="size" if verify == "receipt" else verify)
            if receipt is not None:
                receipts[key] = receipt
                if receipt is not index.get(key):
                    changed.append(receipt)
    if changed:
        with _index_path(state_dir).open("a", encoding="utf-8") as stream:
            stream.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in changed)
    return receipts

## 6. Pipelined download and reporting

`process_download` runs in a worker thread: extract on local disk, copy the kept files into a Drive
staging folder, rename it to `vN/` and write the receipt. `download_sources` writes `progress.json`,
`run_summary.json`, `run_history.jsonl`, `errors.jsonl`, `attempts.jsonl` and `papers.csv` (columns `LOG_FIELDS` from
`00_config`, as read by `04_monitor.ipynb`).

In [ ]:
_OUTPUT_CASE_INSENSITIVE = {}   # output folder -> bool, probed once (Drive calls are slow)


def _output_case_insensitive(output_dir):
    key = str(Path(output_dir).resolve())
    if key not in _OUTPUT_CASE_INSENSITIVE:
        _OUTPUT_CASE_INSENSITIVE[key] = _case_insensitive(output_dir)
    return _OUTPUT_CASE_INSENSITIVE[key]


def process_download(raw, arxiv_id, version, downloaded, output_dir, state_dir, started=None):
    """Extract a fetched archive locally, then publish only kept files to output_dir."""
    started = time.perf_counter() if started is None else started
    raw, output_dir, state_dir = Path(raw), Path(output_dir), Path(state_dir)
    with tempfile.TemporaryDirectory(dir=raw.parent) as local:
        # Extract on fast local disk; rejected archives never touch Drive.
        fold_case = _case_insensitive(local) or _output_case_insensitive(output_dir)
        info = extract_source(raw, Path(local) / "source", fold_case=fold_case)
        raw.unlink(missing_ok=True)  # Free local disk before the Drive copy.
        stage = output_dir / ".staging"
        stage.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=stage) as temporary:
            extracted = Path(temporary) / "source"
            extracted.mkdir()
            for name in info["files"]:
                destination = extracted / name
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(Path(local) / "source" / name, destination)
            target = output_dir / to_folder_id(arxiv_id) / f"v{version}"
            target.parent.mkdir(parents=True, exist_ok=True)
            backup = None
            if target.exists():
                backup = state_dir / "replaced" / f"{to_folder_id(arxiv_id)}_v{version}_{uuid.uuid4().hex}"
                backup.parent.mkdir(parents=True, exist_ok=True)
                os.replace(target, backup)
            try:
                os.replace(extracted, target)
                receipt = dict(info, arxiv_id=arxiv_id, version=version,
                               downloaded_bytes=downloaded, finished_at=_now(),
                               elapsed_seconds=round(time.perf_counter() - started, 3))
                _atomic_json(_receipt_path(state_dir, arxiv_id, version), receipt)
            except Exception:
                if backup is not None and not target.exists():
                    os.replace(backup, target)
                raise
            if backup is not None:
                shutil.rmtree(backup)
            return receipt


def collect_one(client, record, version, output_dir, state_dir):
    """Fetch and process one version synchronously (used by tests and smoke runs)."""
    started = time.perf_counter()
    with tempfile.TemporaryDirectory() as local_temporary:
        raw = Path(local_temporary) / "download.bin"
        downloaded = client.fetch(record["arxiv_id"], version, raw)
        return process_download(raw, record["arxiv_id"], version, downloaded,
                                output_dir, state_dir, started)


class _MemorySampler:
    def __init__(self):
        self.values = []
        self.stop = threading.Event()
        self.thread = threading.Thread(target=self._sample, daemon=True)

    def _sample(self):
        process = psutil.Process() if psutil else None
        while not self.stop.is_set():
            if process:
                self.values.append(process.memory_info().rss)
            self.stop.wait(2)

    def __enter__(self):
        self.thread.start()
        return self

    def __exit__(self, *_):
        self.stop.set()
        self.thread.join()


def _progress(records, receipts):
    total_papers = len(records)
    total_versions = sum(r["version_count"] for r in records)
    completed_versions = len(receipts)
    completed_papers = sum(
        (r["version_count"] == 0 and r.get("status") == "not_found") or
        (r["version_count"] > 0 and all((r["arxiv_id"], v) in receipts for v in range(1, r["version_count"] + 1)))
        for r in records
    )
    return {"papers_completed": completed_papers, "papers_total": total_papers,
            "papers_percent": round(100 * completed_papers / total_papers, 2),
            "versions_completed": completed_versions, "versions_total": total_versions,
            "versions_percent": round(100 * completed_versions / total_versions, 2) if total_versions else 0.0}


def _stamp(after=0.0):
    """Wall-clock tag for log lines, e.g. [11-04 07:56:12], in LOG_TIMEZONE (the runtime clock if unknown),
    `after` seconds from now."""
    try:
        zone = ZoneInfo(LOG_TIMEZONE)
    except Exception:   # no time zone database
        zone = None
    return (datetime.now(zone) + timedelta(seconds=after)).strftime("[%m-%d %H:%M:%S]")


def _rate(bytes_per_second):
    """Download speed for the progress line, e.g. 1.6 MB/s or 14 KB/s."""
    if bytes_per_second >= 1048576:
        return f"{bytes_per_second / 1048576:,.1f} MB/s"
    return f"{bytes_per_second / 1024:,.0f} KB/s"


from html import escape as _escape


def _duration(seconds):
    """Elapsed time or ETA like tqdm, e.g. 7:05:09 or 3:09."""
    minutes, secs = divmod(int(max(0, seconds)), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}:{minutes:02d}:{secs:02d}" if hours else f"{minutes}:{secs:02d}"


class _HtmlBars:
    """download_sources' two bars as HTML, for Colab: Papers (with versions, MB, speed, rate, ETA) and the
    current file (MB done of its size, its speed, MB left).

    One notebook output, redrawn in place through an IPython display handle. Widget bars, and display
    updates sent while the browser is away, are lost when it reconnects to the running cell; so every
    `redisplay` seconds the bars move to a new output (the old one is emptied) and come back within that
    time after a reconnect. Log lines (print) keep streaming above them.
    """

    TRACK = "flex:1;height:14px;border-radius:4px;overflow:hidden;background:rgba(128,128,128,.25)"

    def __init__(self, total, initial, redisplay=300):
        from IPython.display import HTML, display
        self._html, self._display = HTML, display
        self.total, self.initial, self.redisplay = total, initial, redisplay
        self.started = time.monotonic()
        self.handle = None
        self.shown_at = 0.0

    def _row(self, label, fraction, color, text):
        width = 100 * min(max(fraction, 0.0), 1.0)
        return ('<div style="display:flex;align-items:center;gap:8px;margin:3px 0">'
                f'<span style="width:140px;white-space:nowrap;overflow:hidden">{_escape(label)}</span>'
                f'<div style="{self.TRACK}"><div style="width:{width:.2f}%;height:100%;background:{color}"></div></div>'
                f'<span style="min-width:300px;white-space:nowrap">{_escape(text)}</span></div>')

    def draw(self, done, versions_done, versions_total, session_bytes, speed=None, file=None, note=""):
        """file: (label, bytes done, size or None, bytes/s or None) of the current or last transfer;
        speed: average download speed of this run in bytes/s; note: 429 count, waiting for arXiv."""
        elapsed = time.monotonic() - self.started
        rate = (done - self.initial) / elapsed if elapsed > 0 else 0.0
        eta = _duration((self.total - done) / rate) if rate > 0 else "?"
        percent = 100 * done / self.total if self.total else 100.0
        rows = [self._row("Papers", percent / 100, "#43a047",
                          f"{done:,}/{self.total:,} ({percent:.2f}%) | {_duration(elapsed)} < {eta}")]
        if file is None:
            rows.append(self._row("Next file", 0.0, "#1e88e5", "waiting for the first answer"))
        else:
            label, got, size, file_speed = file
            text = f"{got / 1048576:,.2f}/{size / 1048576:,.2f} MB" if size else f"{got / 1048576:,.2f} MB"
            if file_speed:
                text += f" | {_rate(file_speed)}"
            if size:
                text += f" | {(size - got) / 1048576:,.2f} MB left" if got < size else " | done"
            rows.append(self._row(label, got / size if size else 0.0, "#1e88e5", text))
        info = [f"versions {versions_done:,}/{versions_total:,}", f"{session_bytes / 1048576:,.1f} MB downloaded"]
        if speed:
            info.append(f"avg {_rate(speed)}")
        info.append(f"{rate * 3600:,.0f} papers/h")
        if note:
            info.append(note)
        body = (f'<div style="font-family:monospace;font-size:13px">{"".join(rows)}'
                f'<div style="opacity:.75">{_escape(" | ".join(info))}</div></div>')
        now = time.monotonic()
        try:
            if self.handle is None or now - self.shown_at >= self.redisplay:
                if self.handle is not None:
                    self.handle.update(self._html(""))
                self.handle = self._display(self._html(body), display_id=True)
                self.shown_at = now
            else:
                self.handle.update(self._html(body))
        except Exception:
            pass   # a progress bar must never stop the crawl


def _print_progress(progress, say=print):
    say(f"{_stamp()} Papers: {progress['papers_completed']:,}/{progress['papers_total']:,} ({progress['papers_percent']:.2f}%) | "
          f"versions: {progress['versions_completed']:,}/{progress['versions_total']:,} ({progress['versions_percent']:.2f}%)")


def _write_papers_csv(path, records, receipts):
    """Export downloader metrics in the column schema of 04_monitor.ipynb."""
    path = Path(path)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        with temporary.open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=LOG_FIELDS)
            writer.writeheader()
            for record in records:
                identifier, count = record["arxiv_id"], record["version_count"]
                found = [receipts[(identifier, version)] for version in range(1, count + 1)
                         if (identifier, version) in receipts]
                status = (record.get("status", "error") if count == 0 else
                          "ok" if len(found) == count and all(r["status"] == "ok" for r in found) else
                          "partial" if found else "error")
                writer.writerow({"id": identifier, "versions": count,
                                 "versions_ok": sum(r["status"] == "ok" for r in found),
                                 "size_before": sum(r["before_bytes"] for r in found),
                                 "size_after": sum(r["after_bytes"] for r in found),
                                 "status": status,
                                 "seconds": round(sum(r.get("elapsed_seconds", 0) for r in found), 3),
                                 "error": "unprocessed source versions" if len(found) < count else "",
                                 "logged_at": _now()})
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _process_job(job_dir, *args):
    try:
        return process_download(Path(job_dir) / "download.bin", *args)
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)


_FATAL_ERRNOS = {code for code in (errno.ENOSPC, getattr(errno, "EDQUOT", None), errno.ENOTCONN, errno.EROFS) if code}


def _is_fatal(exc):
    """A storage error that every later version would hit too: disk or Drive quota full, Drive unmounted."""
    for _ in range(10):   # follow the chain of causes
        if exc is None:
            return False
        if isinstance(exc, OSError) and exc.errno in _FATAL_ERRNOS:
            return True
        exc = exc.__cause__ or exc.__context__
    return False


class RunLock:
    """Refuse to run while another runtime works on the same folder (best effort, also on Drive).

    The lock file holds the owner (host:pid) and a heartbeat refreshed every `beat` seconds. It is taken
    over when the heartbeat is older than `stale` seconds (the other Colab runtime is gone) or, on the same
    host, when the owner's process no longer exists (a restarted kernel). Re-entrant within one process.
    """

    _held = {}   # lock path -> [depth, stop event, heartbeat thread]

    def __init__(self, path, beat=60, stale=180):
        self.path, self.beat, self.stale = Path(path), beat, stale
        self.owner = f"{socket.gethostname()}:{os.getpid()}"

    def _read(self):
        try:
            return json.loads(self.path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return None

    def _write(self):
        _atomic_json(self.path, {"owner": self.owner, "heartbeat": time.time(), "heartbeat_utc": _now()})

    def acquire(self):
        key = str(self.path.resolve())
        if key in RunLock._held:
            RunLock._held[key][0] += 1
            return self
        other = self._read()
        if other and other.get("owner") != self.owner:
            age = time.time() - float(other.get("heartbeat", 0))
            host, _, pid = str(other.get("owner", "")).rpartition(":")
            gone = (host == socket.gethostname() and pid.isdigit()
                    and psutil is not None and not psutil.pid_exists(int(pid)))
            if age < self.stale and not gone:
                raise RuntimeError(
                    f"Another run ({other.get('owner')}) is working on {self.path.parent} "
                    f"(last heartbeat {age:.0f} s ago). Stop it first. If it was a Colab runtime that no longer "
                    f"exists, run again in {self.stale - age:.0f} s, or delete {self.path}.")
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._write()
        stop = threading.Event()

        def heartbeat():
            while not stop.wait(self.beat):
                try:
                    self._write()
                except OSError:
                    pass

        thread = threading.Thread(target=heartbeat, daemon=True)
        thread.start()
        RunLock._held[key] = [1, stop, thread]
        return self

    def release(self):
        key = str(self.path.resolve())
        held = RunLock._held.get(key)
        if not held:
            return
        held[0] -= 1
        if held[0] > 0:
            return
        held[1].set()
        held[2].join(timeout=5)
        del RunLock._held[key]
        if (self._read() or {}).get("owner") == self.owner:
            self.path.unlink(missing_ok=True)

    def __enter__(self):
        return self.acquire()

    def __exit__(self, *exc_info):
        self.release()


def _free_bytes(folder):
    try:
        return shutil.disk_usage(folder).free
    except OSError:
        return None


def check_free_space(records, receipts, output_dir, temp_dir, workers):
    """Problems (empty list if none) if the output or temp folder cannot hold the rest of the crawl."""
    mb = 1024 * 1024
    remaining = sum(1 for r in records for v in range(1, r["version_count"] + 1) if (r["arxiv_id"], v) not in receipts)
    kept = [r["after_bytes"] for r in receipts.values() if r["status"] == "ok"]
    per_version = sum(kept) / len(kept) if kept else 300 * 1024
    needs = ((f"output folder {output_dir}", output_dir, int(remaining * per_version * 1.5) + 50 * mb),
             (f"local temp folder {temp_dir}", temp_dir, 2 * workers * 200 * mb))
    problems = []
    for label, folder, need in needs:
        free = _free_bytes(folder)
        if remaining and free is not None and free < need:
            problems.append(f"{label}: {free / mb:,.0f} MB free, about {need / mb:,.0f} MB needed")
    return problems


def _clean_leftovers(output_dir, state_dir):
    """Staging folders and backups left by a killed run (safe: RunLock keeps other downloaders out)."""
    removed = 0
    for folder in (Path(output_dir) / ".staging", Path(state_dir) / "replaced"):
        if folder.is_dir():
            for child in folder.iterdir():
                if child.is_dir():
                    shutil.rmtree(child, ignore_errors=True)
                else:
                    child.unlink(missing_ok=True)
                removed += 1
    return removed


def download_sources(manifest_path, output_dir, state_dir, *, user_agent=USER_AGENT,
        delay=ARXIV_DELAY, session=None, sleeper=time.sleep, workers=2, verify="size",
        temp_dir=None, progress=False, max_consecutive_failures=20, check_space=True):
    """Run or resume the downloader; errors are logged and retried next run.

    One paced connection fetches archives to local temp storage while up to
    ``workers`` threads extract them and write .tex/.bib to ``output_dir``.
    ``verify='receipt'`` trusts receipts.jsonl at startup (no saved file is read); ``'size'`` checks
    every saved file by size and ``'sha256'`` re-hashes it (hours on Drive for the full corpus).
    ``progress=True`` shows two bars: papers done out of every discovered paper (with versions,
    MB downloaded, average speed and ETA), and the file being downloaded with its speed and the MB left.
    On Colab (``HTML_BARS``) they are HTML (``_HtmlBars``), drawn again in a new output every 5 minutes
    so they survive a browser reconnect. With text bars (``TEXT_BARS``) it is one line: the file and
    its MB left follow the paper counts.

    Safety: a lock keeps a second runtime out of ``state_dir``; leftovers of a killed run are removed;
    the free space is checked before the first request (``check_space``); and the run stops early
    (``stopped_reason`` in the summary) on a storage error (disk/Drive full, Drive unmounted) or after
    ``max_consecutive_failures`` failed versions in a row, instead of failing every remaining version.
    A 404/410 (arXiv has no source for that version, e.g. withdrawn) does not count towards that stop.
    """
    if workers < 1:
        raise ValueError("workers must be positive")
    if verify not in ("receipt", "size", "sha256"):
        raise ValueError("verify must be 'receipt', 'size' or 'sha256'")
    start = time.perf_counter()
    records = load_manifest(manifest_path)
    output_dir, state_dir = Path(output_dir), Path(state_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    state_dir.mkdir(parents=True, exist_ok=True)
    lock = RunLock(state_dir / "download.lock").acquire()
    try:
        return _download_locked(records, output_dir, state_dir, start, user_agent, delay, session, sleeper,
                                workers, verify, temp_dir, progress, max_consecutive_failures, check_space)
    finally:
        lock.release()


def _download_locked(records, output_dir, state_dir, start, user_agent, delay, session, sleeper,
                     workers, verify, temp_dir, progress, max_consecutive_failures, check_space):
    removed = _clean_leftovers(output_dir, state_dir)
    if removed:
        print(f"Removed {removed} leftover staging folder(s)/backup(s) of an interrupted run")
    receipts = load_receipts(records, output_dir, state_dir, verify=verify)
    print(f"Progress at start/resume (verified in {time.perf_counter() - start:.1f}s):")
    last_printed = _progress(records, receipts)
    _print_progress(last_printed)
    if check_space:
        problems = check_free_space(records, receipts, output_dir, temp_dir or tempfile.gettempdir(), workers)
        if problems:
            raise RuntimeError("Not enough free space; free some space (or pass check_space=False):\n  "
                               + "\n  ".join(problems))
    # Each run starts with the versions never requested. Versions requested in an earlier run but not
    # finished (404, throttled, cut, or the run stopped mid-download) come after them, the least-tried
    # first. records keeps the manifest order for papers.csv and the progress numbers.
    tries = {key: n for key, n in _past_tries(state_dir).items() if key not in receipts}
    queue = sorted(records, key=lambda r: max((tries.get((r["arxiv_id"], v), 0)
                                               for v in range(1, r["version_count"] + 1)), default=0))
    if tries:
        print(f"{len(tries):,} unfinished versions from earlier runs are retried after the new ones")
    skipped = failed = consecutive = 0
    stop_reason = None
    current_bytes = sum(r["after_bytes"] for r in receipts.values())
    disk_peak_estimate = current_bytes
    client = SourceClient(session=session, user_agent=user_agent, delay=delay, sleeper=sleeper)

    hinted = False
    throttles = {"count": 0, "line_at": None}   # answers 429 this run, time of the last log line about them

    def show_throttle(label, seconds):
        """A 429 is not a failure (the same version is asked again after the wait), so it is not logged one by
        one: the Papers line counts them, and one log line sums them up at most every 10 minutes."""
        nonlocal hinted
        throttles["count"] += 1
        now = time.monotonic()
        if throttles["line_at"] is None or now - throttles["line_at"] >= 600:
            throttles["line_at"] = now
            say(f"{_stamp()} HTTP 429 (arXiv rate limit): {throttles['count']:,} so far this run; the same versions "
                f"are asked again, requests now {client.pace:,.0f} s apart")
        if seconds >= client.max_cooldown and not hinted:
            hinted = True
            say("arXiv keeps refusing this runtime (HTTP 429): its IP is probably rate-limited. Runtime > "
                "Disconnect and delete runtime gives a new one; the crawl resumes where it stopped.")

    client.on_throttle = show_throttle

    def show_wait(seconds):
        """Cooldown or back-off before the next request: say so on the Papers line (else it looks stuck)."""
        nonlocal file_note
        file_note = f" | waiting {seconds:,.0f} s for arXiv, until {_stamp(seconds)[7:-1]}"
        refresh_bars()

    owns_session = session is None
    interrupted = False
    max_pending = 2 * workers  # Bounds local disk used by fetched, unprocessed archives.
    pending = {}
    pending_bytes = {}          # future -> size of its raw archive, still on local disk
    local_bytes = local_peak = 0
    work_root = Path(tempfile.mkdtemp(prefix="arxiv-src-", dir=temp_dir))
    index_stream = _index_path(state_dir).open("a", encoding="utf-8")
    attempts_stream = (state_dir / "attempts.jsonl").open("a", encoding="utf-8")

    counts = {r["arxiv_id"]: r["version_count"] for r in records}
    versions_total = sum(counts.values())
    completed = {pid for pid, n in counts.items()
                 if n > 0 and all((pid, v) in receipts for v in range(1, n + 1))}
    session_bytes = 0
    paper_bar = file_bar = html_bars = None
    file_state = None   # HTML bars: (label, bytes done, size, bytes/s) of the current or last file
    transfer = {"bytes": 0, "seconds": 0.0}   # streamed this run and the time it took: the average speed
    if progress and HTML_BARS:
        html_bars = _HtmlBars(total=sum(n > 0 for n in counts.values()), initial=len(completed))
    elif progress and tqdm is not None:
        paper_bar = tqdm(total=sum(n > 0 for n in counts.values()), initial=len(completed),
                         unit="paper", desc="Papers", position=0, dynamic_ncols=True, file=sys.stdout)
        if not TEXT_BARS:
            file_bar = tqdm(total=0, unit="B", unit_scale=True, unit_divisor=1024, desc="Next file",
                            position=1, leave=False, dynamic_ncols=True)
    say = tqdm.write if paper_bar is not None else print
    if paper_bar is not None or html_bars is not None:
        client.on_wait = show_wait
    file_note = ""      # text bars: " | <id>vN: X MB left" while a file downloads; " | waiting ..." for arXiv
    last_drawn = 0.0

    def refresh_bars(force=True):
        nonlocal last_drawn
        if (paper_bar is None and html_bars is None) or (not force and time.monotonic() - last_drawn < 1.0):
            return   # while a file downloads, the bars are redrawn at most once a second
        last_drawn = time.monotonic()
        speed = transfer["bytes"] / transfer["seconds"] if transfer["seconds"] > 0 else None
        throttled = f" | 429 x{throttles['count']:,}" if throttles["count"] else ""
        if html_bars is not None:
            html_bars.draw(len(completed), len(receipts), versions_total, session_bytes, speed, file_state,
                           (throttled + file_note).removeprefix(" | "))
            return
        paper_bar.set_postfix_str(f"versions {len(receipts):,}/{versions_total:,} | "
                                  f"{session_bytes / 1048576:,.0f} MB downloaded"
                                  + (f" at {_rate(speed)}" if speed else "") + throttled + file_note)

    def file_progress(label):
        """Callback for SourceClient.fetch: the current file, its size, the MB left and the speed."""
        first = last = None   # (time, bytes) of the first and the previous callback of this transfer

        def update(done, total):
            nonlocal file_note, file_state, first, last
            now = time.monotonic()
            if last is None or done < last[1]:   # a new transfer (or one that starts over)
                first = last = (now, done)
            transfer["bytes"] += done - last[1]
            transfer["seconds"] += now - last[0]
            last = (now, done)
            elapsed = now - first[0]
            speed = (done - first[1]) / elapsed if elapsed > 0 else None
            finished = bool(total) and done >= total
            if html_bars is not None:
                file_note = ""   # a transfer is running: drop a waiting note
                file_state = (label, done, total, speed)
                refresh_bars(force=done == first[1] or finished)
                return
            if file_bar is None:   # text bars: shown after the paper counts on the Papers line
                mb = (total - done if total else done) / 1048576
                rate = f" at {_rate(speed)}" if speed and elapsed >= 1 else ""
                file_note = ("" if finished
                             else f" | {label}: {mb:,.1f} MB {'left' if total else 'so far'}{rate}")
                refresh_bars(force=done == 0 or not file_note)
                return
            if file_note:   # widget bars show the speed themselves; clear a waiting note
                file_note = ""
                refresh_bars()
            if done == 0:
                file_bar.reset(total=total)
                file_bar.set_description(label, refresh=False)
            if total:   # set before update(), so the bar is drawn with the current value
                file_bar.set_postfix_str(f"{(total - done) / 1048576:,.1f} MB left", refresh=False)
            file_bar.update(done - file_bar.n)
        return update

    def stop_for_storage(exc):
        nonlocal stop_reason
        if stop_reason is None:
            stop_reason = f"storage error, every later version would fail too: {exc!r}"

    def log_failure(identifier, version, exc):
        nonlocal failed, consecutive, stop_reason
        failed += 1
        # A 404/410 means arXiv has no source for this version (e.g. withdrawn), not an outage; after a
        # resume these versions are retried in a row at the end, so counting them would stop the run.
        if getattr(getattr(exc, "response", None), "status_code", None) not in (404, 410):
            consecutive += 1
        try:
            with (state_dir / "errors.jsonl").open("a", encoding="utf-8") as errors:
                errors.write(json.dumps({"arxiv_id": identifier, "version": version,
                                         "error": repr(exc), "at": _now()}, ensure_ascii=False) + "\n")
        except OSError as write_error:
            stop_for_storage(write_error)
        say(f"{_stamp()} FAILED {identifier}v{version}: {exc}")
        if _is_fatal(exc):
            stop_for_storage(exc)
        elif stop_reason is None and consecutive >= max_consecutive_failures:
            stop_reason = (f"{consecutive} versions failed in a row (last: {exc!r}); arXiv, the network "
                           f"or Google Drive is probably down")

    def harvest(block):
        nonlocal current_bytes, disk_peak_estimate, local_bytes, consecutive
        if not pending:
            return
        done = wait(pending, return_when=FIRST_COMPLETED)[0] if block else [f for f in pending if f.done()]
        for future in done:
            key = pending.pop(future)
            local_bytes -= pending_bytes.pop(future, 0)
            try:
                receipt = future.result()
            except Exception as exc:
                log_failure(*key, exc)
                continue
            consecutive = 0
            disk_peak_estimate = max(disk_peak_estimate,
                                     current_bytes + receipt["downloaded_bytes"] + receipt["after_bytes"])
            current_bytes += receipt["after_bytes"]
            receipts[key] = receipt
            try:
                index_stream.write(json.dumps(receipt, ensure_ascii=False) + "\n")
                index_stream.flush()
            except OSError as exc:
                stop_for_storage(exc)
            pid = key[0]
            if pid not in completed and all((pid, v) in receipts for v in range(1, counts[pid] + 1)):
                completed.add(pid)
                if paper_bar is not None:
                    paper_bar.update(1)
            refresh_bars()

    try:
        with _MemorySampler() as memory, ThreadPoolExecutor(max_workers=workers) as pool:
            try:
                # Every version in queue order. One answered 429 goes back to the end of this run's work (up to
                # THROTTLE_REQUEUES times), so the crawl moves on to the next version instead of waiting.
                work = deque((r["arxiv_id"], v) for r in queue for v in range(1, r["version_count"] + 1))
                requeued = {}
                popped = 0
                while work and not stop_reason:
                    key = work.popleft()
                    identifier, version = key
                    popped += 1
                    if key in receipts:
                        skipped += 1
                    else:
                        harvest(block=False)
                        while len(pending) >= max_pending:
                            harvest(block=True)
                        if stop_reason:
                            break
                        try:   # written before the request: a run stopped or killed mid-download counts it
                            attempts_stream.write(json.dumps({"arxiv_id": identifier, "version": version,
                                                              "at": _now()}) + "\n")
                            attempts_stream.flush()
                        except OSError as exc:
                            stop_for_storage(exc)
                            break
                        file_note = ""   # drop a waiting note left by the previous request
                        started = time.perf_counter()
                        job = Path(tempfile.mkdtemp(dir=work_root))
                        try:
                            downloaded = client.fetch(
                                identifier, version, job / "download.bin",
                                progress=file_progress(f"{identifier}v{version}") if paper_bar is not None or html_bars is not None else None)
                        except Throttled:
                            shutil.rmtree(job, ignore_errors=True)
                            requeued[key] = requeued.get(key, 0) + 1
                            if requeued[key] <= THROTTLE_REQUEUES:
                                work.append(key)   # later in this run; after that, in the next run
                            if client.throttled >= client.max_throttled:
                                stop_reason = (f"arXiv answered HTTP 429 to {client.throttled} requests in a row: this "
                                               "runtime's IP is rate-limited. Runtime > Disconnect and delete runtime, "
                                               "then run again")
                        except Exception as exc:
                            shutil.rmtree(job, ignore_errors=True)
                            log_failure(identifier, version, exc)
                            if client.pause:
                                say(f"{_stamp()} arXiv is throttling or cutting downloads: next request in {client.pause:.0f} s")
                        except BaseException:
                            shutil.rmtree(job, ignore_errors=True)
                            raise
                        else:
                            future = pool.submit(_process_job, job, identifier, version, downloaded,
                                                 output_dir, state_dir, started)
                            pending[future] = key
                            pending_bytes[future] = downloaded
                            local_bytes += downloaded
                            local_peak = max(local_peak, local_bytes)
                            session_bytes += downloaded
                            refresh_bars()
                    if popped % 100 == 0 or not work:
                        harvest(block=False)
                        current = _progress(records, receipts)
                        if current != last_printed:   # papers skipped on resume change nothing
                            _print_progress(current, say)
                            last_printed = current
                            try:
                                _atomic_json(state_dir / "progress.json", current)
                            except OSError as exc:
                                stop_for_storage(exc)
            except KeyboardInterrupt:
                interrupted = True
                say("Interrupted; finishing archives already downloaded, then saving receipts.")
            finally:
                while pending:  # Archives already fetched are never thrown away.
                    harvest(block=True)
    finally:
        if html_bars is not None:
            file_note = ""
            refresh_bars()   # the final numbers stay on screen
        for bar in (file_bar, paper_bar):
            if bar is not None:
                bar.close()
        for stream in (index_stream, attempts_stream):
            try:
                stream.close()
            except OSError:
                pass
        shutil.rmtree(work_root, ignore_errors=True)
        if owns_session:
            client.session.close()
    progress = _progress(records, receipts)
    final_bytes = sum(r["after_bytes"] for r in receipts.values())
    summary = {"finished_at": _now(), "elapsed_seconds": round(time.perf_counter() - start, 3),
               "interrupted": interrupted, "stopped_reason": stop_reason, "progress": progress,
               "workers": workers, "verify": verify, "kept_extensions": sorted(KEEP),
               "elapsed_per_assigned_paper_seconds": round((time.perf_counter() - start) / len(records), 6),
               "versions_ok": sum(r["status"] == "ok" for r in receipts.values()),
               "versions_empty": sum(r["status"] == "empty_source" for r in receipts.values()),
               "papers_all_sources": sum(
                   r["version_count"] > 0 and all(
                       (r["arxiv_id"], v) in receipts and receipts[(r["arxiv_id"], v)]["status"] == "ok"
                       for v in range(1, r["version_count"] + 1)) for r in records),
               "versions_skipped": skipped, "versions_failed_this_run": failed,
               "source_before_bytes": sum(r["before_bytes"] for r in receipts.values()),
               "source_after_bytes": final_bytes,
               "output_bytes": final_bytes,
               "disk_peak_estimate_bytes": max(disk_peak_estimate, final_bytes),
               "local_temp_peak_bytes": local_peak,   # raw archives waiting on local disk at once
               "ram_peak_bytes": max(memory.values, default=None),
               "ram_average_bytes": round(sum(memory.values) / len(memory.values)) if memory.values else None}
    try:
        _atomic_json(state_dir / "run_summary.json", summary)
        _atomic_json(state_dir / "progress.json", progress)
        _write_papers_csv(state_dir / "papers.csv", records, receipts)
        with (state_dir / "run_history.jsonl").open("a", encoding="utf-8") as history:
            history.write(json.dumps(summary, ensure_ascii=False) + "\n")
            history.flush()
            os.fsync(history.fileno())
    except OSError as exc:
        print(f"Could not write the run summary: {exc!r}")
    _print_progress(progress)
    if stop_reason:
        print(f"\nSTOPPED EARLY: {stop_reason}\nFix the cause (free space on Drive, re-mount Drive, wait for "
              f"arXiv), then run again: finished versions are kept and the failed ones are retried.")
    return summary

## 7. Offline tests

These cells check the downloader with temporary files and fake HTTP sessions; they do not contact
arXiv. The test classes are always defined but run only when `RUN_TEST` is `True` (see `00_config`).

In [ ]:
"""Offline contract tests for the member-2 source downloader."""

import contextlib
import gzip
import csv
import io
import json
from pathlib import Path
import sys
import tarfile
import tempfile
import unittest
from unittest.mock import patch
import zipfile



TEX = b"\\documentclass{article}\n\\begin{figure}\\includegraphics{plot}\n"


def tar_bytes(files):
    out = io.BytesIO()
    with tarfile.open(fileobj=out, mode="w:gz") as archive:
        for name, content in files:
            info = tarfile.TarInfo(name)
            info.size = len(content)
            archive.addfile(info, io.BytesIO(content))
    return out.getvalue()


class FakeResponse:
    def __init__(self, content, status=200, headers=None):
        self.content = content
        self.status_code = status
        self.headers = headers or {}

    def iter_content(self, chunk_size):
        yield self.content

    def close(self):
        pass


class CutResponse(FakeResponse):
    """Sends its content, then the connection breaks (like arXiv's IncompleteRead)."""

    def iter_content(self, chunk_size):
        yield self.content
        raise requests.exceptions.ChunkedEncodingError("Connection broken: IncompleteRead")


class FakeSession:
    def __init__(self, responses):
        self.responses = iter(responses)
        self.calls = []
        self.headers = {}

    def get(self, url, **kwargs):
        self.calls.append((url, kwargs))
        return next(self.responses)

    def close(self):
        pass



class DownloadTestBase(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.root = Path(self.temp.name)


### 7a. Manifest and request pacing

Validate discovery records, assigned ranges, retries, and the 3-second request gate.

In [ ]:
class ManifestAndPacingTests(DownloadTestBase):
    def test_manifest_accepts_group_discovery_schema(self):
        path = self.root / "manifest.json"
        path.write_text(json.dumps({"papers": [
            {"id": "2603.20580", "versions": 2, "title": "sample", "status": "ok"},
            {"id": "2603.20581", "versions": 0, "status": "not_found"},
        ]}), encoding="utf-8")
        records = load_manifest(path)
        self.assertEqual(records[0]["arxiv_id"], "2603.20580")
        self.assertEqual(records[0]["version_count"], 2)
        self.assertEqual(records[0]["title"], "sample")
        self.assertEqual(records[1]["version_count"], 0)

    def test_manifest_rejects_duplicate_and_bad_version_count(self):
        for records in (
            [{"arxiv_id": "2603.20580", "version_count": 1}] * 2,
            [{"id": "2603.20580", "versions": 0, "status": "ok"}],
            [{"arxiv_id": "2603.00000", "version_count": 1}],
        ):
            with self.subTest(records=records), self.assertRaises(ValueError):
                validate_manifest(records)

    def test_discovery_error_with_no_versions_stays_pending(self):
        records = validate_manifest([{"id": "2603.20580", "versions": 0, "status": "error"}])
        self.assertEqual(records[0]["version_count"], 0)
        self.assertEqual(_progress(records, {})["papers_percent"], 0.0)

    def test_429_retry_after_and_request_pacing(self):
        session = FakeSession([FakeResponse(b"busy", 429, {"Retry-After": "45"}), FakeResponse(b"abc")])
        current = [0.0]
        def sleep(seconds):
            current[0] += seconds
        starts = []
        original_get = session.get
        def observed_get(url, **kwargs):
            starts.append(current[0])
            return original_get(url, **kwargs)
        session.get = observed_get
        client = SourceClient(session, "CourseBot/1.0 (contact@example.org)", clock=lambda: current[0], sleeper=sleep,
                              delay=3.0)
        target = self.root / "raw.bin"
        with self.assertRaises(Throttled):   # one request, no retry inside fetch
            client.fetch("2603.20580", 2, target)
        self.assertEqual(client.fetch("2603.20580", 2, target), 3)
        self.assertEqual(starts, [0.0, 45.0])   # Retry-After
        self.assertEqual(target.read_bytes(), b"abc")
        self.assertEqual([c[0] for c in session.calls], ["https://export.arxiv.org/src/2603.20580v2"] * 2)
        self.assertEqual(session.headers["User-Agent"], "CourseBot/1.0 (contact@example.org)")

    def test_429_raises_throttled_and_cools_down_after_5_in_a_row(self):
        session = FakeSession([FakeResponse(b"busy", 429) for _ in range(6)] + [FakeResponse(b"ok")])
        seen = []
        client = SourceClient(session, sleeper=lambda s: None, delay=3.0)
        client.on_throttle = lambda label, wait: seen.append(wait)
        for version in range(1, 7):   # one request each: a 429 is never retried inside fetch
            with self.assertRaises(Throttled):
                client.fetch("2603.20580", version, self.root / "raw")
        self.assertEqual(len(session.calls), 6)
        self.assertEqual(seen, [0.0, 0.0, 0.0, 0.0, 10.0, 20.0])   # a cooldown from the 5th 429 in a row
        self.assertEqual(client.fetch("2603.20580", 7, self.root / "raw"), 2)
        self.assertEqual((client.throttled, client.cooldown), (0, 0.0))

    def test_cut_download_resumes_with_range(self):
        session = FakeSession([CutResponse(b"abc", 200, {"Content-Length": "6", "ETag": '"v1"'}),
                               FakeResponse(b"def", 206, {"Content-Length": "3", "Content-Range": "bytes 3-5/6"})])
        client = SourceClient(session, sleeper=lambda s: None)
        target = self.root / "raw.bin"
        self.assertEqual(client.fetch("2603.20580", 1, target), 6)
        self.assertEqual(target.read_bytes(), b"abcdef")
        self.assertEqual(session.calls[0][1]["headers"], {})
        self.assertEqual(session.calls[1][1]["headers"], {"Range": "bytes=3-", "If-Range": '"v1"'})
        self.assertEqual(list(self.root.glob("*.part")), [])

    def test_full_answer_to_a_range_request_starts_over(self):
        session = FakeSession([CutResponse(b"abc", 200, {"ETag": '"v1"'}), FakeResponse(b"abcdef", 200)])
        target = self.root / "raw.bin"
        self.assertEqual(SourceClient(session, sleeper=lambda s: None).fetch("2603.20580", 1, target), 6)
        self.assertEqual(target.read_bytes(), b"abcdef")   # the file changed: no stale prefix

    def test_cooldown_after_throttled_versions_doubles_and_resets(self):
        session = FakeSession([FakeResponse(b"busy", 429), FakeResponse(b"busy", 429), FakeResponse(b"ok"),
                               CutResponse(b"ab", 200)])
        now, starts = [0.0], []
        original = session.get
        def get(url, **kwargs):
            starts.append(now[0])
            return original(url, **kwargs)
        session.get = get
        client = SourceClient(session, max_attempts=1, clock=lambda: now[0],
                              sleeper=lambda s: now.__setitem__(0, now[0] + s), delay=3.0)
        for _ in range(2):   # 429, 429: the caller would move on; here the same version is asked again
            with self.assertRaises(Throttled):
                client.fetch("2603.20580", 1, self.root / "raw")
        self.assertEqual(client.fetch("2603.20580", 1, self.root / "raw"), 2)
        self.assertEqual((client.cooldown, client.throttled, client.pause), (0.0, 0, 0.0))
        self.assertAlmostEqual(client.pace, 4.0)   # 3 s -> 6 -> 8 (the cap) apart, then halved after the success
        with self.assertRaises(requests.RequestException):     # a cut connection still fails and cools down
            client.fetch("2603.20580", 2, self.root / "raw")
        self.assertEqual(client.pause, 10.0)
        self.assertEqual(starts, [0.0, 6.0, 14.0, 18.0])   # no cooldown for two 429s, only the wider pace

    def test_long_waits_are_reported_and_speed_is_readable(self):
        now, waits = [0.0], []
        client = SourceClient(FakeSession([]), clock=lambda: now[0],
                              sleeper=lambda s: now.__setitem__(0, now[0] + s), delay=3.0)
        client.on_wait = waits.append
        client._pace()
        client._pace()          # the normal 3 s delay is not reported
        client._cool_down()
        client._pace()
        self.assertEqual(waits, [10.0])
        self.assertEqual((_rate(14 * 1024), _rate(1.6 * 1048576)), ("14 KB/s", "1.6 MB/s"))

    def test_nonretryable_404_is_sent_once(self):
        session = FakeSession([FakeResponse(b"missing", 404)])
        client = SourceClient(session, sleeper=lambda seconds: None)
        with self.assertRaises(Exception):
            client.fetch("2603.20580", 1, self.root / "raw")
        self.assertEqual(len(session.calls), 1)

    def test_assigned_manifest_coverage_rejects_truncated_cache(self):
        self.assertEqual(len(assigned_ids()), 20000)
        complete = [{"id": identifier, "versions": 1} for identifier in assigned_ids()]
        self.assertEqual(len(validate_assigned_manifest(complete)), 20000)
        with self.assertRaises(ValueError):
            validate_assigned_manifest([{"id": "2603.20580", "versions": 1}])


### 7b. Archive filtering and safety

Check exact TeX bytes, that only `.tex`/`.bib` files are written, PDF-only versions, and unsafe archive entries.

In [ ]:
class ArchiveAndSafetyTests(DownloadTestBase):
    def test_tar_keeps_tex_bytes_and_skips_figures(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("sub/main.tex", TEX), ("sub/refs.bib", b"@book{a}"), ("sub/pic.PNG", b"image"), ("sub/plot.pdf", b"%PDF")]))
        target = self.root / "v1"
        info = extract_source(raw, target)
        self.assertEqual(info["status"], "ok")
        self.assertEqual((target / "sub/main.tex").read_bytes(), TEX)
        self.assertEqual({p.suffix for p in target.rglob("*") if p.is_file()}, {".tex", ".bib"})
        self.assertEqual(info["before_bytes"], len(TEX) + len(b"@book{a}") + 5 + 4)
        self.assertEqual(info["after_bytes"], len(TEX) + len(b"@book{a}"))

    def test_gzip_single_tex_and_pdf_empty(self):
        raw = self.root / "raw"
        raw.write_bytes(gzip.compress(TEX))
        self.assertEqual(extract_source(raw, self.root / "v1")["status"], "ok")
        self.assertEqual((self.root / "v1/main.tex").read_bytes(), TEX)
        raw.write_bytes(b"%PDF-1.7\nxxx")
        info = extract_source(raw, self.root / "v2")
        self.assertEqual(info["status"], "empty_source")
        self.assertEqual(list((self.root / "v2").iterdir()), [])

    def test_single_tex_after_long_comment_is_kept(self):
        content = (b"% comment\n" * 1200) + TEX
        raw = self.root / "raw"
        for payload, folder in ((content, "plain"), (gzip.compress(content), "gz")):
            raw.write_bytes(payload)
            info = extract_source(raw, self.root / folder)
            self.assertEqual(info["status"], "ok")
            self.assertEqual((self.root / folder / "main.tex").read_bytes(), content)

    def test_unrecognized_success_body_is_not_cached_as_empty(self):
        raw = self.root / "raw"
        for payload in (b"<html>temporary gateway error</html>", b"nonsense bytes"):
            raw.write_bytes(payload)
            with self.subTest(payload=payload), self.assertRaises(ValueError):
                extract_source(raw, self.root / "v1")

    def test_single_gzip_source_is_bounded_during_decompression(self):
        raw = self.root / "raw"
        raw.write_bytes(gzip.compress(TEX + b"x" * 100))
        with (patch.dict(globals(), MAX_TEXT_BYTES=len(TEX) + 10),
              patch.object(tempfile, "TemporaryFile", side_effect=AssertionError("oversized source staged")),
              self.assertRaises(ValueError)):
            extract_source(raw, self.root / "v1")

    def test_unsafe_tar_is_rejected(self):
        raw = self.root / "raw"
        for members in ([("../escape.tex", TEX)], [("main.tex", TEX), ("main.tex/x.tex", TEX)]):
            raw.write_bytes(tar_bytes(members))
            with self.subTest(members=[m[0] for m in members]), self.assertRaises(ValueError):
                extract_source(raw, self.root / "unsafe")
        self.assertFalse((self.root / "escape.tex").exists())

    def test_case_variant_names_are_both_kept(self):
        # Real case: 2609.32867v1 has sections/limitations.tex and sections/Limitations.tex
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("main.tex", TEX), ("sections/limitations.tex", b"lower"),
                                   ("sections/Limitations.tex", b"upper")]))
        for fold_case, folder in ((None, "auto"), (True, "folded")):
            info = extract_source(raw, self.root / folder, fold_case=fold_case)
            self.assertEqual(info["status"], "ok")
            stored = sorted((self.root / folder / name).read_bytes() for name in info["files"])
            self.assertEqual(stored, sorted([TEX, b"lower", b"upper"]))
            self.assertEqual(info["before_bytes"], len(TEX) + 10)
        self.assertEqual(info["renamed"], {"sections/Limitations~2.tex": "sections/Limitations.tex"})
        zipped = io.BytesIO()
        with zipfile.ZipFile(zipped, "w") as archive:
            archive.writestr("main.tex", TEX)
            archive.writestr("A.tex", b"upper")
            archive.writestr("a.tex", b"lower")
        raw.write_bytes(zipped.getvalue())
        info = extract_source(raw, self.root / "zip", fold_case=True)
        self.assertEqual(sorted(info["files"]), ["A.tex", "a~2.tex", "main.tex"])

    def test_repeated_tar_name_keeps_last_copy(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("main.tex", b"\\documentclass{old}"), ("main.tex", TEX)]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual((self.root / "v1/main.tex").read_bytes(), TEX)
        self.assertEqual(info["before_bytes"], len(TEX))

    def test_zip_symlink_is_rejected(self):
        out = io.BytesIO()
        with zipfile.ZipFile(out, "w") as archive:
            archive.writestr("main.tex", TEX)
            member = zipfile.ZipInfo("link.tex")
            member.create_system = 3
            member.external_attr = 0o120777 << 16
            archive.writestr(member, "main.tex")
        raw = self.root / "raw"
        raw.write_bytes(out.getvalue())
        with self.assertRaises(ValueError):
            extract_source(raw, self.root / "v1")

    def test_tar_with_dot_prefix_is_accepted(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("./main.tex", TEX), ("./fig.png", b"image")]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual(info["status"], "ok")
        self.assertEqual((self.root / "v1/main.tex").read_bytes(), TEX)

    def test_archive_download_uses_runtime_temp_not_output_drive(self):
        output, state = self.root / "output", self.root / "state"
        seen = []
        class Client:
            def fetch(self, identifier, version, destination):
                seen.append(Path(destination))
                payload = tar_bytes([("main.tex", TEX), ("image.png", b"large image")])
                Path(destination).write_bytes(payload)
                return len(payload)
        collect_one(Client(), {"arxiv_id": "2603.20580"}, 1, output, state)
        self.assertFalse(seen[0].is_relative_to(output))
        self.assertEqual((output / "2603-20580/v1/main.tex").read_bytes(), TEX)


### 7c. Resume and reporting

Check interruption recovery, receipts, progress percentages, and reporting files.

In [ ]:
class ResumeAndReportingTests(DownloadTestBase):
    def test_interruption_preserves_previous_version_receipt(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"id": "2603.20580", "versions": 2}]), encoding="utf-8")
        class InterruptSession(FakeSession):
            def get(self, url, **kwargs):
                if len(self.calls) == 1:
                    raise KeyboardInterrupt()
                return super().get(url, **kwargs)
        first = download_sources(manifest, self.root / "out", self.root / "state",
                             session=InterruptSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]),
                             sleeper=lambda s: None)
        self.assertTrue(first["interrupted"])
        self.assertEqual(first["progress"]["versions_percent"], 50.0)
        later = download_sources(manifest, self.root / "out", self.root / "state",
                             session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]),
                             sleeper=lambda s: None)
        self.assertEqual(later["versions_skipped"], 1)
        self.assertEqual(later["progress"]["versions_percent"], 100.0)

    def test_resume_verifies_receipt_and_reports_percent(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"arxiv_id": "2603.20580", "version_count": 2}]), encoding="utf-8")
        out, state = self.root / "output", self.root / "state"
        session = FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)])), FakeResponse(b"%PDF-1.7\n")])
        first = download_sources(manifest, out, state, session=session, delay=3, sleeper=lambda s: None)
        self.assertEqual(first["progress"]["papers_percent"], 100.0)
        self.assertEqual(first["progress"]["versions_percent"], 100.0)
        self.assertEqual(first["versions_ok"], 1)
        self.assertEqual(first["versions_empty"], 1)
        with (state / "papers.csv").open(newline="", encoding="utf-8") as stream:
            row = next(csv.DictReader(stream))
        self.assertEqual(row["id"], "2603.20580")
        self.assertEqual(row["versions"], "2")
        self.assertEqual(row["versions_ok"], "1")
        self.assertEqual(row["size_after"], str(len(TEX)))
        self.assertEqual(row["status"], "partial")
        again = download_sources(manifest, out, state, session=FakeSession([]), delay=3, sleeper=lambda s: None)
        self.assertEqual(again["versions_skipped"], 2)
        (out / "2603-20580/v1/main.tex").write_bytes(b"corrupt")
        repaired = download_sources(manifest, out, state, session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]), delay=3, sleeper=lambda s: None)
        self.assertEqual(repaired["versions_skipped"], 1)
        self.assertEqual((out / "2603-20580/v1/main.tex").read_bytes(), TEX)

    def test_resume_check_skips_versions_without_receipt(self):
        state = self.root / "state"
        (state / "receipts" / "2603-20582").mkdir(parents=True)          # per-version receipt files
        (state / "receipts.jsonl").write_text(json.dumps({"arxiv_id": "2603.20580", "version": 1}) + "\n",
                                             encoding="utf-8")
        records = [{"arxiv_id": f"2603.{20580 + i}", "version_count": 2} for i in range(4)]
        checked = []
        def spy(state_dir, output_dir, arxiv_id, version, **kwargs):
            checked.append((arxiv_id, version))
        with patch.dict(globals(), verified_receipt=spy):
            self.assertEqual(load_receipts(records, self.root / "output", state), {})
        self.assertEqual(checked, [("2603.20580", 1), ("2603.20580", 2), ("2603.20582", 1), ("2603.20582", 2)])

    def test_resume_prints_progress_only_when_it_changes(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"id": f"2603.{20580 + i:05d}", "versions": 1} for i in range(200)]),
                            encoding="utf-8")
        out, state = self.root / "output", self.root / "state"
        download_sources(manifest, out, state, sleeper=lambda s: None,
                         session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)])) for _ in range(200)]))
        printed = io.StringIO()
        with contextlib.redirect_stdout(printed):
            again = download_sources(manifest, out, state, session=FakeSession([]), sleeper=lambda s: None)
        self.assertEqual(again["versions_skipped"], 200)
        lines = [line for line in printed.getvalue().splitlines() if " Papers: " in line]
        self.assertEqual(len(lines), 2)   # at start and at the end, none for the 200 skipped papers
        self.assertRegex(lines[0], r"^\[\d{2}-\d{2} \d{2}:\d{2}:\d{2}\] Papers: ")

    def test_receipt_mode_trusts_the_index_and_checks_the_rest_on_disk(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"id": "2603.20580", "versions": 2}]), encoding="utf-8")
        out, state = self.root / "output", self.root / "state"
        download_sources(manifest, out, state, sleeper=lambda s: None,
                         session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)])) for _ in range(2)]))
        (out / "2603-20580/v1/main.tex").write_bytes(b"corrupt")   # not noticed: the index is trusted
        index = state / "receipts.jsonl"
        v1_only = [line for line in index.read_text(encoding="utf-8").splitlines() if json.loads(line)["version"] == 1]
        index.write_text(v1_only[-1] + "\n", encoding="utf-8")   # workers finish in any order
        trusted = download_sources(manifest, out, state, verify="receipt", session=FakeSession([]),
                                   sleeper=lambda s: None)
        self.assertEqual(trusted["versions_skipped"], 2)        # v2 from its per-version receipt file
        repaired = download_sources(manifest, out, state, verify="size", sleeper=lambda s: None,
                                    session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]))
        self.assertEqual(repaired["versions_skipped"], 1)       # the size check finds the damage
        self.assertEqual((out / "2603-20580/v1/main.tex").read_bytes(), TEX)

    def test_failed_version_is_retried_on_next_run(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"arxiv_id": "2603.20580", "version_count": 1}]), encoding="utf-8")
        output, state = self.root / "output", self.root / "state"
        failed = download_sources(manifest, output, state, session=FakeSession([FakeResponse(b"missing", 404)]), sleeper=lambda s: None)
        self.assertEqual(failed["progress"]["versions_percent"], 0.0)
        self.assertEqual(failed["versions_failed_this_run"], 1)
        recovered = download_sources(manifest, output, state, session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]), sleeper=lambda s: None)
        self.assertEqual(recovered["progress"]["versions_percent"], 100.0)
        self.assertGreaterEqual(recovered["disk_peak_estimate_bytes"], recovered["output_bytes"])
        history = (state / "run_history.jsonl").read_text(encoding="utf-8").splitlines()
        self.assertEqual(len(history), 2)
        self.assertEqual([json.loads(line)["progress"]["versions_percent"] for line in history], [0.0, 100.0])

    def test_versions_that_failed_before_are_retried_last(self):
        manifest = self.root / "manifest.json"
        output, state = self.root / "output", self.root / "state"
        manifest.write_text(json.dumps([{"arxiv_id": "2603.20580", "version_count": 1}]), encoding="utf-8")
        download_sources(manifest, output, state, session=FakeSession([FakeResponse(b"missing", 404)]),
                         sleeper=lambda s: None)
        manifest.write_text(json.dumps([{"arxiv_id": "2603.20580", "version_count": 1},
                                        {"arxiv_id": "2603.20581", "version_count": 1}]), encoding="utf-8")
        session = FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)])) for _ in range(2)])
        summary = download_sources(manifest, output, state, session=session, sleeper=lambda s: None)
        self.assertEqual([url.rsplit("/", 1)[1] for url, _ in session.calls], ["2603.20581v1", "2603.20580v1"])
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)
        with (state / "papers.csv").open(newline="", encoding="utf-8") as stream:
            self.assertEqual([row["id"] for row in csv.DictReader(stream)], ["2603.20580", "2603.20581"])

    def test_429_moves_on_and_asks_again_later(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"arxiv_id": "2603.20580", "version_count": 1},
                                        {"arxiv_id": "2603.20581", "version_count": 1}]), encoding="utf-8")
        state, printed = self.root / "state", io.StringIO()
        ok = lambda: FakeResponse(tar_bytes([("main.tex", TEX)]))
        session = FakeSession([FakeResponse(b"busy", 429), ok(), ok()])
        with contextlib.redirect_stdout(printed):
            summary = download_sources(manifest, self.root / "output", state, session=session, sleeper=lambda s: None)
        self.assertEqual([url.rsplit("/", 1)[1] for url, _ in session.calls],
                         ["2603.20580v1", "2603.20581v1", "2603.20580v1"])   # moved on, then asked again
        self.assertEqual((summary["versions_failed_this_run"], summary["progress"]["versions_percent"]), (0, 100.0))
        self.assertFalse((state / "errors.jsonl").exists())
        self.assertIn("HTTP 429 (arXiv rate limit): 1 so far this run", printed.getvalue())

    def test_run_stops_after_max_throttled_429_in_a_row(self):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"arxiv_id": f"2603.2058{i}", "version_count": 1} for i in range(3)]),
                            encoding="utf-8")
        original = globals()["SourceClient"]
        globals()["SourceClient"] = lambda **kwargs: original(max_throttled=2, **kwargs)
        try:
            summary = download_sources(manifest, self.root / "output", self.root / "state", sleeper=lambda s: None,
                                       session=FakeSession([FakeResponse(b"busy", 429) for _ in range(5)]))
        finally:
            globals()["SourceClient"] = original
        self.assertIn("HTTP 429 to 2 requests in a row", summary["stopped_reason"])
        self.assertEqual(summary["versions_failed_this_run"], 0)

    def test_versions_tried_before_go_last_least_tried_first(self):
        class Session(FakeSession):   # KeyboardInterrupt in the list: the run is stopped mid-request
            def get(self, url, **kwargs):
                response = super().get(url, **kwargs)
                if response is KeyboardInterrupt:
                    raise KeyboardInterrupt
                return response

        def run(ids, responses):
            manifest = self.root / "manifest.json"
            manifest.write_text(json.dumps([{"arxiv_id": i, "version_count": 1} for i in ids]), encoding="utf-8")
            session = Session(responses)
            summary = download_sources(manifest, self.root / "output", self.root / "state", session=session,
                                       sleeper=lambda s: None)
            return [url.rsplit("/", 1)[1][:-2] for url, _ in session.calls], summary

        a, b, c = "2603.20580", "2603.20581", "2603.20582"
        ok = lambda: FakeResponse(tar_bytes([("main.tex", TEX)]))
        self.assertEqual(run([a, b], [KeyboardInterrupt])[0], [a])                   # stopped: no error logged
        self.assertEqual(run([a, b], [FakeResponse(b"missing", 404), KeyboardInterrupt])[0], [b, a])
        calls, summary = run([a, b, c], [ok(), ok(), ok()])
        self.assertEqual(calls, [c, b, a])    # never tried, then tried once (b), then twice (a)
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)

### 7d. Filtering and pipelined download

Check that only `.tex`/`.bib` are kept, that older `.bbl`/`.sty`/`.cls` files are pruned on resume, that the next request is sent while the previous archive is still being processed, and that resume works from the `receipts.jsonl` index.

In [ ]:
import errno
import shutil
import threading
import time


class FilteringAndPerformanceTests(DownloadTestBase):
    def write_manifest(self, versions=1):
        manifest = self.root / "manifest.json"
        manifest.write_text(json.dumps([{"id": "2603.20580", "versions": versions}]), encoding="utf-8")
        return manifest

    def test_only_tex_and_bib_are_kept(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("main.tex", TEX), ("refs.bib", b"@book{a}"), ("main.bbl", b"bbl"),
                                   ("style.sty", b"sty"), ("class.cls", b"cls"), ("fig.png", b"image")]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual(sorted(info["files"]), ["main.tex", "refs.bib"])
        self.assertEqual(sorted(p.name for p in (self.root / "v1").rglob("*") if p.is_file()), ["main.tex", "refs.bib"])

    def test_bib_without_tex_is_empty_source(self):
        raw = self.root / "raw"
        raw.write_bytes(tar_bytes([("refs.bib", b"@book{a}"), ("paper.pdf", b"%PDF")]))
        info = extract_source(raw, self.root / "v1")
        self.assertEqual(info["status"], "empty_source")
        self.assertEqual(list((self.root / "v1").rglob("*")), [])

    def test_resume_prunes_legacy_bbl_without_new_request(self):
        manifest, out, state = self.write_manifest(), self.root / "out", self.root / "state"
        folder = out / "2603-20580/v1"
        (folder / "sub").mkdir(parents=True)
        (folder / "main.tex").write_bytes(TEX)
        (folder / "sub/main.bbl").write_bytes(b"legacy")
        legacy = {"arxiv_id": "2603.20580", "version": 1, "status": "ok", "before_bytes": 100,
                  "after_bytes": len(TEX) + 6, "downloaded_bytes": 50,
                  "files": {"main.tex": {"size": len(TEX), "sha256": hashlib.sha256(TEX).hexdigest()},
                            "sub/main.bbl": {"size": 6, "sha256": hashlib.sha256(b"legacy").hexdigest()}}}
        _atomic_json(_receipt_path(state, "2603.20580", 1), legacy)
        summary = download_sources(manifest, out, state, session=FakeSession([]), sleeper=lambda s: None)
        self.assertEqual(summary["versions_skipped"], 1)
        self.assertEqual(summary["source_after_bytes"], len(TEX))
        self.assertEqual([p.name for p in folder.rglob("*")], ["main.tex"])
        again = download_sources(manifest, out, state, session=FakeSession([]), sleeper=lambda s: None)
        self.assertEqual(again["versions_skipped"], 1)

    def test_next_fetch_overlaps_previous_processing(self):
        manifest, out, state = self.write_manifest(2), self.root / "out", self.root / "state"
        events, second_fetch = [], threading.Event()
        class Session(FakeSession):
            def get(self, url, **kwargs):
                events.append("fetch " + url[-2:])
                if url.endswith("v2"):
                    second_fetch.set()
                return super().get(url, **kwargs)
        original = process_download
        def slow_process(raw, identifier, version, *args):
            if version == 1:
                second_fetch.wait(5)
            receipt = original(raw, identifier, version, *args)
            events.append(f"processed v{version}")
            return receipt
        session = Session([FakeResponse(tar_bytes([("main.tex", TEX)])), FakeResponse(tar_bytes([("main.tex", TEX)]))])
        with patch.dict(globals(), process_download=slow_process):
            summary = download_sources(manifest, out, state, session=session, sleeper=lambda s: None)
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)
        self.assertLess(events.index("fetch v2"), events.index("processed v1"))
        # both raw archives were on local disk at the same time
        self.assertEqual(summary["local_temp_peak_bytes"], 2 * len(tar_bytes([("main.tex", TEX)])))

    def test_progress_callback_and_bars(self):
        payload = tar_bytes([("main.tex", TEX)])
        seen = []
        client = SourceClient(FakeSession([FakeResponse(payload, headers={"Content-Length": str(len(payload))})]),
                              sleeper=lambda s: None)
        client.fetch("2603.20580", 1, self.root / "raw", progress=lambda done, total: seen.append((done, total)))
        self.assertEqual((seen[0], seen[-1]), ((0, len(payload)), (len(payload), len(payload))))
        manifest, out, state = self.write_manifest(2), self.root / "out", self.root / "state"
        summary = download_sources(manifest, out, state, progress=True, sleeper=lambda s: None,
                                   session=FakeSession([FakeResponse(payload), FakeResponse(payload)]))
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)

    def test_resume_uses_receipt_index_and_sha256_mode_detects_same_size_damage(self):
        manifest, out, state = self.write_manifest(), self.root / "out", self.root / "state"
        download_sources(manifest, out, state, session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]),
                     sleeper=lambda s: None)
        shutil.rmtree(state / "receipts")  # Index alone is enough for a fast resume.
        with (state / "receipts.jsonl").open("a", encoding="utf-8") as stream:
            stream.write('{"torn": ')
        self.assertEqual(download_sources(manifest, out, state, session=FakeSession([]),
                                      sleeper=lambda s: None)["versions_skipped"], 1)
        (out / "2603-20580/v1/main.tex").write_bytes(b"x" * len(TEX))
        self.assertEqual(download_sources(manifest, out, state, session=FakeSession([]),
                                      sleeper=lambda s: None)["versions_skipped"], 1)
        repaired = download_sources(manifest, out, state, verify="sha256", sleeper=lambda s: None,
                                session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]))
        self.assertEqual(repaired["versions_skipped"], 0)
        self.assertEqual((out / "2603-20580/v1/main.tex").read_bytes(), TEX)

class RobustnessTests(DownloadTestBase):
    def manifest(self, versions):
        path = self.root / "manifest.json"
        path.write_text(json.dumps([{"id": f"2603.{20580 + i:05d}", "versions": 1} for i in range(versions)]),
                        encoding="utf-8")
        return path

    def test_stops_after_consecutive_failures(self):
        session = FakeSession([FakeResponse(b"busy", 503) for _ in range(40)])   # 4 attempts per version
        summary = download_sources(self.manifest(10), self.root / "out", self.root / "state", session=session,
                                   sleeper=lambda s: None, max_consecutive_failures=3)
        self.assertIn("3 versions failed in a row", summary["stopped_reason"])
        self.assertEqual(len(session.calls), 12)           # the other 7 versions were not requested

    def test_missing_sources_do_not_stop_the_run(self):
        responses = [FakeResponse(b"missing", 404) for _ in range(9)] + [FakeResponse(tar_bytes([("main.tex", TEX)]))]
        summary = download_sources(self.manifest(10), self.root / "out", self.root / "state", sleeper=lambda s: None,
                                   session=FakeSession(responses), max_consecutive_failures=3)
        self.assertIsNone(summary["stopped_reason"])     # nine 404s in a row are not an outage
        self.assertEqual(summary["versions_failed_this_run"], 9)
        self.assertEqual(summary["progress"]["versions_completed"], 1)

    def test_storage_full_stops_at_once(self):
        def full_disk(*args):
            raise OSError(errno.ENOSPC, "No space left on device")
        payload = tar_bytes([("main.tex", TEX)])
        session = FakeSession([FakeResponse(payload) for _ in range(10)])
        with patch.dict(globals(), process_download=full_disk):
            summary = download_sources(self.manifest(10), self.root / "out", self.root / "state", workers=1,
                                       session=session, sleeper=lambda s: None)
        self.assertIn("storage error", summary["stopped_reason"])
        self.assertLessEqual(len(session.calls), 3)        # only the archives already in flight

    def test_lock_blocks_a_second_runtime(self):
        state = self.root / "state"
        state.mkdir()
        lock = state / "download.lock"
        lock.write_text(json.dumps({"owner": "other-host:1", "heartbeat": time.time()}), encoding="utf-8")
        with self.assertRaises(RuntimeError):
            download_sources(self.manifest(1), self.root / "out", state, session=FakeSession([]), sleeper=lambda s: None)
        lock.write_text(json.dumps({"owner": "other-host:1", "heartbeat": time.time() - 3600}), encoding="utf-8")
        summary = download_sources(self.manifest(1), self.root / "out", state, sleeper=lambda s: None,
                                   session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]))
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)
        self.assertFalse(lock.exists())                    # released at the end

    def test_leftovers_of_a_killed_run_are_removed(self):
        out, state = self.root / "out", self.root / "state"
        (out / ".staging" / "tmpabc" / "source").mkdir(parents=True)
        (state / "replaced" / "2603-20580_v1_x").mkdir(parents=True)
        download_sources(self.manifest(1), out, state, sleeper=lambda s: None,
                         session=FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))]))
        self.assertEqual(list((out / ".staging").iterdir()), [])
        self.assertEqual(list((state / "replaced").iterdir()), [])

    def test_free_space_is_checked_before_any_request(self):
        session = FakeSession([FakeResponse(tar_bytes([("main.tex", TEX)]))])
        tiny = type("Usage", (), {"free": 1024})()
        with patch.object(shutil, "disk_usage", return_value=tiny), self.assertRaises(RuntimeError):
            download_sources(self.manifest(1), self.root / "out", self.root / "state", session=session,
                             sleeper=lambda s: None)
        self.assertEqual(session.calls, [])
        with patch.object(shutil, "disk_usage", return_value=tiny):
            summary = download_sources(self.manifest(1), self.root / "out", self.root / "state", session=session,
                                       sleeper=lambda s: None, check_space=False)
        self.assertEqual(summary["progress"]["versions_percent"], 100.0)

### 7e. Run the offline tests

Runs the 39 tests when `RUN_TEST` is `True`. A simulated 404 prints a `FAILED` line and the
interruption test prints `Interrupted`; the final `OK` is the pass criterion.

In [ ]:
if RUN_TEST:
    suite = unittest.TestSuite()
    for case in (ManifestAndPacingTests, ArchiveAndSafetyTests, ResumeAndReportingTests,
                 FilteringAndPerformanceTests, RobustnessTests):
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise AssertionError("Downloader offline tests failed")

## 8. Output and resume

Sources: `MyDrive/arxiv-project/data/raw/<yymm-id>/vN/` (only `.tex`/`.bib`). Receipts (one file per
version plus the `receipts.jsonl` index), `errors.jsonl`, `attempts.jsonl`, `progress.json`, `papers.csv`,
`run_summary.json` and `run_history.jsonl`: `state/download/`. After a Colab disconnect, run
`05_run_pipeline.ipynb` again: the crawl resumes from the receipts. Use one downloader at a time per
Drive folder.